# Train a YOLO26 Object Detection Model (Roboflow dataset)

Works for any labelled dataset exported from Roboflow in **YOLO26** format.

**Before starting:** Runtime > Change runtime type > **T4 GPU**.

**Important:** free Colab deletes everything when it disconnects or sits idle. This notebook saves the trained model and results to **Google Drive**, so they are never lost. Keep the tab open while training.

## 1. Settings
Change these for each new dataset or training run. `RUN_NAME` becomes the folder name in Drive, so use a new name for each run (e.g. `practice_v1`, `practice_v2`).

In [ ]:
WORKSPACE = "YOUR_WORKSPACE"      # from the Roboflow download code
PROJECT   = "YOUR_PROJECT"        # project ID (lowercase, with dashes)
VERSION   = 1                     # Roboflow dataset version
MODEL     = "yolo26n.pt"          # n = nano (fast); try yolo26s.pt for better accuracy
EPOCHS    = 30
RUN_NAME  = "practice_v1"         # folder name for this run in Drive
SAVE_DIR  = "/content/drive/MyDrive/yolo_runs"   # no Drive space? use "/content/runs" and download the model after training

## 2. Connect Google Drive
Click **Allow** when asked. This access is only for your session; it is not saved in the notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Check the GPU
Should show a **Tesla T4**.

In [ ]:
!nvidia-smi

## 4. Install Ultralytics and Roboflow

In [ ]:
!pip install -U ultralytics roboflow -q
import ultralytics
ultralytics.checks()

## 5. Download the dataset from Roboflow
The API key is read from **Colab Secrets** (key icon on the left). Add a secret named `ROBOFLOW_API_KEY` and turn on **Notebook access**. Never type the key directly into code.

In [ ]:
from google.colab import userdata
from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolo26")
!cat {dataset.location}/data.yaml

## 6. Train
About 10 to 20 minutes on a T4. **Keep this tab open and your laptop awake.**

- **Epochs:** one full pass over the training images. 30 is usually enough; the score stops improving after about 20.
- **imgsz:** 640 is standard. Try 960 or 1024 later for very small parts.

Results are saved straight to Drive: `yolo_runs/<RUN_NAME>`.

In [ ]:
from ultralytics import YOLO
model = YOLO(MODEL)
results = model.train(data=f"{dataset.location}/data.yaml", epochs=EPOCHS, imgsz=640,
                      project=SAVE_DIR, name=RUN_NAME, exist_ok=True)
!ls "{SAVE_DIR}/{RUN_NAME}/weights"

### 6b. Download the trained model now
Downloads `best.pt` to your computer straight away, so it is safe even if Colab disconnects. Essential when `SAVE_DIR` is not on Google Drive.

In [ ]:
from google.colab import files
files.download(f"{SAVE_DIR}/{RUN_NAME}/weights/best.pt")

## 7. Load the trained model
Run this after training, **or any time later** (even after a disconnect) to load the saved model from Drive. If Colab disconnected, run cells 1 and 2 first.

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
run = f"{SAVE_DIR}/{RUN_NAME}"
best = YOLO(f"{run}/weights/best.pt")
print("Loaded:", run)

## 8. Check the results
The table at the end of training shows per class:
- **P (precision):** of the boxes drawn, how many were correct
- **R (recall):** of the real objects, how many were found
- **mAP50:** overall score. Above 0.8 is good, 0.5 to 0.8 okay, below 0.5 needs better labels or more data.

Note: if validation frames come from the same videos as training frames, the score is optimistic. Always test on new videos too (cell 10).

In [ ]:
for f in ["results.png", "confusion_matrix.png", "val_batch0_pred.jpg"]:
    print(f); display(Image(filename=f"{run}/{f}", width=800))

## 9. Test on the test images
Runs the model on the test split and shows the first 10 results. Needs the dataset from cell 5.

In [ ]:
import glob
preds = best.predict(source=f"{dataset.location}/test/images", save=True, conf=0.4,
                     project=run, name="test_images", exist_ok=True)
for p in glob.glob(f"{run}/test_images/*.jpg")[:10]:
    display(Image(filename=p, height=400))

## 10. Test on your own video
Put a test video (one **not** used for training) in `My Drive/yolo_test_videos/`, set its name below and run. The result is saved in Drive under `runs/<RUN_NAME>/test_video/`.

In [ ]:
VIDEO = "/content/drive/MyDrive/yolo_test_videos/your_test_video.mp4"   # or upload to /content and use "/content/your_test_video.mp4"
best.predict(source=VIDEO, save=True, conf=0.4, project=run, name="test_video", exist_ok=True)
!ls "{run}/test_video"

## Before sharing this notebook
**Edit > Clear all outputs**, then save. This keeps paths, links and logs out of the shared file.